# No RL 01 - Classical Baseline

Use the existing alpha-beta engine without a neural model. Evaluation coefficients can be tuned through games in notebook 03. Default path: **01 -> 03 -> 04 -> 05**. Optional supervised path: **01 -> 02 -> 03 -> 04 -> 05**. Use a CPU runtime here.

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project Setup

Uses `configs/no_rl.yaml`. No league, self-play, PPO, or DQN is run. Colab's installed PyTorch is preserved.

In [ ]:
from pathlib import Path
import sys
import subprocess
from IPython.display import display
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

def show_figure(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl/non_rl.py").is_file():
    raise FileNotFoundError(f"Place the updated project contents directly in {PROJECT_ROOT}")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                       str(PROJECT_ROOT / "requirements_colab.txt")])
sys.path.insert(0, str(PROJECT_ROOT))
from chess_rl.non_rl import load_no_rl_config
from chess_rl.reproducibility import read_json, sha256
cfg = load_no_rl_config(PROJECT_ROOT)
print("Run:", cfg["run_id"], "| Variant:", cfg["non_rl"]["variant"])
print("Root:", PROJECT_ROOT)


## Choose the Variant

For no neural training, leave `non_rl.variant: classical` in `configs/no_rl.yaml`. For supervised learning without RL, set it to `supervised` and change `run_id` to a new name such as `no_rl_supervised_v1`. Keep the same configuration throughout one run. Do not change the original `default.yaml` to select this workflow.

In [ ]:
print("Settings file:", PROJECT_ROOT / "configs/no_rl.yaml")
print(cfg["search"])

## What the Baseline Knows

The current evaluator uses material, piece-square tables, attacked-square activity, bishop pairs, doubled/isolated pawns, and passed pawns. Iterative deepening, alpha-beta, quiescence, and a transposition table provide calculation. This notebook does not add opening books or implement every strategy from a chess textbook.

## Inspect a Position

The board is standard chess, not a reduced grid environment. The classical evaluator returns a score from the side-to-move perspective. The displayed position is an illustrative input, not a benchmark.

In [ ]:
import chess
from chess_rl.classical_evaluation import evaluate_cp, evaluation_breakdown
from chess_rl.search import SearchEngine
from chess_rl.non_rl import variant_config

board = chess.Board()
for uci in ("e2e4", "e7e5", "g1f3"):
    board.push_uci(uci)
display(board)
print("Legal moves:", board.legal_moves.count())
print("Static evaluation (centipawn units):", evaluate_cp(board, cfg["search"]["evaluation_weights"]))
for row in evaluation_breakdown(board, cfg["search"]["evaluation_weights"]):
    print(f'{row["feature"]:22s} {row["value"]:7g} x {row["weight"]:7g} = {row["contribution_cp"]:8.2f} cp')

## Search for a Move

This runs the classical search directly. A legal fallback is available before deeper search. The board must be unchanged when search returns. This is ordinary correctness work, not submission validation.

In [ ]:
classical_cfg = variant_config(cfg, "classical")
engine = SearchEngine(classical_cfg["search"])
before = board.fen()
result = engine.choose(board, time_left_ms=2000)
assert result.move in board.legal_moves
assert board.fen() == before
assert engine.neural is None
print("Move:", result.move.uci(), "| Completed depth:", result.depth,
      "| Nodes:", result.nodes, "| Time (ms):", round(result.elapsed_ms, 2))
after = board.copy()
after.push(result.move)
display(after)

## Reserve Opening Suites

The same immutable development/held-out fixtures are shared with the research project. Their legal random playout provenance is recorded; they are not balanced or the platform's hidden opening set.

In [ ]:
from chess_rl.dataset import prepare_openings
opening_manifest = prepare_openings(PROJECT_ROOT, seed=cfg["seed"])
print(opening_manifest)

## Check Workflow Correctness

Bounded tests verify the independent classical path and its stage dependencies. They do not start full training, full tournaments, or submission checks.

In [ ]:
subprocess.check_call([sys.executable, "-m", "pytest", "-q", "tests/test_non_rl.py",
                       "tests/test_evaluation_tuning.py"],
                      cwd=PROJECT_ROOT)

## Next Notebook

Classical: open **03_evaluate_no_rl.ipynb**. Supervised: open **02_optional_supervised.ipynb**. Do not run the original RL notebook 03 for this workflow.